# 📕 Advanced & Cross-Cutting ML Topics (Revision Notes)

This is the 4th notebook — covers everything that didn't fit cleanly into the core
Regression / Classification / Unsupervised notebooks: neural network basics, extra Naive Bayes
variants, probability calibration, statistical model comparison, regularization paths, advanced
encoding, proper pipelines, model interpretability (SHAP), extra clustering algorithms, ICA/Factor
Analysis, a hands-on data leakage demo, and a minimal deployment example.

**Datasets used:** `titanic` and `tips` from seaborn (reused so you can compare against notebooks 1 & 2).


In [ ]:
# If any of these are missing, uncomment and run:
# !pip install shap scikit-learn seaborn pandas numpy scipy category_encoders flask --quiet


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.linear_model import LogisticRegression, Ridge, Lasso, LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.calibration import CalibratedClassifierCV, calibration_curve

from sklearn.cluster import SpectralClustering, MeanShift, estimate_bandwidth, KMeans
from sklearn.decomposition import PCA, FactorAnalysis, FastICA
from sklearn.neighbors import NearestNeighbors

from sklearn.metrics import (accuracy_score, f1_score, precision_recall_curve, roc_auc_score,
                              mean_squared_error, r2_score, brier_score_loss)

from scipy import stats

import warnings
warnings.filterwarnings("ignore")

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("shap not installed -> pip install shap")

try:
    import category_encoders as ce
    CE_AVAILABLE = True
except ImportError:
    CE_AVAILABLE = False
    print("category_encoders not installed -> pip install category_encoders")

sns.set_style("whitegrid")


In [ ]:
titanic = sns.load_dataset('titanic').drop(columns=['deck','embark_town','alive','class','who','adult_male'])
titanic['age'] = SimpleImputer(strategy='median').fit_transform(titanic[['age']])
titanic['embarked'] = titanic['embarked'].fillna(titanic['embarked'].mode()[0])

tips = sns.load_dataset('tips')
titanic.head()

## 2. Neural Network Basics (MLP)
**Why:** Multi-Layer Perceptrons are the simplest neural networks — a stack of fully-connected layers
with non-linear activations. Good to know before moving to deep learning frameworks (PyTorch/TensorFlow).
They need SCALED input and usually more data than classical ML models to shine.

In [ ]:
# Classification MLP on Titanic
titanic_enc = pd.get_dummies(titanic, columns=['sex','embarked'], drop_first=True)
X = titanic_enc.drop(columns=['survived'])
y = titanic_enc['survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

# hidden_layer_sizes=(16,8) means 2 hidden layers with 16 and 8 neurons.
# activation='relu' is the standard non-linearity. max_iter raised since MLP needs more epochs to converge.
mlp_clf = MLPClassifier(hidden_layer_sizes=(16,8), activation='relu', max_iter=1000, random_state=42)
mlp_clf.fit(X_train_s, y_train)
print("MLP Classifier accuracy:", accuracy_score(y_test, mlp_clf.predict(X_test_s)))

In [ ]:
# Regression MLP on tips
tips_enc = pd.get_dummies(tips, columns=['sex','smoker','day','time'], drop_first=True)
Xr = tips_enc.drop(columns=['total_bill'])
yr = tips_enc['total_bill']
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.2, random_state=42)

scaler_r = StandardScaler()
Xr_train_s = scaler_r.fit_transform(Xr_train)
Xr_test_s = scaler_r.transform(Xr_test)

mlp_reg = MLPRegressor(hidden_layer_sizes=(16,8), max_iter=2000, random_state=42)
mlp_reg.fit(Xr_train_s, yr_train)
print("MLP Regressor R2:", r2_score(yr_test, mlp_reg.predict(Xr_test_s)))

## 3. Naive Bayes Variants
**Why different variants:** the "right" Naive Bayes depends on your feature type:
- **GaussianNB**: continuous features, assumes each feature is normally distributed per class.
- **MultinomialNB**: count/frequency data (e.g. word counts in text classification / bag-of-words).
- **BernoulliNB**: binary/boolean features (e.g. word present/absent, or one-hot flags).


In [ ]:
# GaussianNB - continuous features (age, fare)
gnb = GaussianNB().fit(X_train[['age','fare']], y_train)
print("GaussianNB accuracy:", accuracy_score(y_test, gnb.predict(X_test[['age','fare']])))

# MultinomialNB - needs non-negative counts, so we discretize fare into bins as a stand-in for "counts"
fare_bins_train = pd.cut(X_train['fare'], bins=5, labels=False).values.reshape(-1,1)
fare_bins_test = pd.cut(X_test['fare'], bins=5, labels=False).values.reshape(-1,1)
mnb = MultinomialNB().fit(fare_bins_train, y_train)
print("MultinomialNB accuracy (fare bins as counts):", accuracy_score(y_test, mnb.predict(fare_bins_test)))

# BernoulliNB - binary features (e.g. sex_male flag, is_alone)
binary_cols = [c for c in X_train.columns if X_train[c].nunique() <= 2]
bnb = BernoulliNB().fit(X_train[binary_cols], y_train)
print("BernoulliNB accuracy (binary cols only):", accuracy_score(y_test, bnb.predict(X_test[binary_cols])))

## 4. Threshold Tuning & Probability Calibration

### 4.1 Threshold Tuning
**Why:** by default, classifiers use 0.5 as the cutoff between classes. But if False Negatives are more
costly than False Positives (or vice versa), you should move the threshold to optimize for your actual
business goal (e.g. recall-focused for disease detection) rather than accuracy.

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)
probs = rf.predict_proba(X_test)[:,1]

precision, recall, thresholds = precision_recall_curve(y_test, probs)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-9)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]

print(f"Default threshold (0.5) F1: {f1_score(y_test, (probs>=0.5).astype(int)):.3f}")
print(f"Best threshold ({best_threshold:.2f}) F1: {f1_scores[best_idx]:.3f}")

plt.figure(figsize=(6,4))
plt.plot(thresholds, precision[:-1], label='Precision')
plt.plot(thresholds, recall[:-1], label='Recall')
plt.axvline(best_threshold, color='red', linestyle='--', label='Best F1 threshold')
plt.xlabel('Threshold'); plt.legend()
plt.title('Precision/Recall vs Decision Threshold')
plt.show()

### 4.2 Probability Calibration
**Why:** a model can be accurate at CLASSIFYING but still have unreliable predicted PROBABILITIES
(e.g. it says "90% confident" but is only right 70% of the time). This matters whenever you actually
use the probability score downstream (risk scoring, ranking, thresholding).
A **calibration curve** plots predicted probability vs actual observed frequency — a perfectly
calibrated model follows the diagonal. `CalibratedClassifierCV` fixes poorly calibrated models.

In [ ]:
prob_true, prob_pred = calibration_curve(y_test, probs, n_bins=10)

plt.figure(figsize=(5,5))
plt.plot(prob_pred, prob_true, 'o-', label='Random Forest (uncalibrated)')
plt.plot([0,1],[0,1],'k--', label='Perfectly calibrated')
plt.xlabel('Mean predicted probability'); plt.ylabel('Fraction of positives')
plt.title('Calibration Curve')
plt.legend()
plt.show()

print("Brier score (lower=better calibrated):", brier_score_loss(y_test, probs))

# Fix calibration using Platt scaling / isotonic regression
calibrated = CalibratedClassifierCV(RandomForestClassifier(n_estimators=200, random_state=42), method='sigmoid', cv=5)
calibrated.fit(X_train, y_train)
calibrated_probs = calibrated.predict_proba(X_test)[:,1]
print("Brier score after calibration:", brier_score_loss(y_test, calibrated_probs))

## 5. Statistical Significance Between Models
**Why:** "Model A got 82% accuracy, Model B got 80%" — is that a REAL difference or just noise from
the random train/test split? We use a paired t-test across cross-validation folds to check if the
difference in performance is statistically significant (p < 0.05 conventionally).

In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

scores_rf = cross_val_score(RandomForestClassifier(n_estimators=200, random_state=42), X, y, cv=skf, scoring='accuracy')
scores_lr = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=skf, scoring='accuracy')

t_stat, p_value = stats.ttest_rel(scores_rf, scores_lr)
print(f"Random Forest mean acc: {scores_rf.mean():.3f} | Logistic Regression mean acc: {scores_lr.mean():.3f}")
print(f"Paired t-test: t={t_stat:.3f}, p={p_value:.3f}")
print("Statistically significant difference" if p_value < 0.05 else "NOT statistically significant - difference could be noise")

## 6. Regularization Path Visualization
**Why:** instead of picking one alpha value blindly, plotting how each coefficient shrinks as
regularization strength (alpha) increases builds intuition for what Ridge/Lasso are actually doing.
Notice how Lasso coefficients can hit exactly zero (feature selection), while Ridge only shrinks
them smoothly toward zero.

In [ ]:
alphas = np.logspace(-3, 2, 50)
ridge_coefs, lasso_coefs = [], []

Xr_train_scaled = StandardScaler().fit_transform(Xr_train)
for a in alphas:
    ridge_coefs.append(Ridge(alpha=a).fit(Xr_train_scaled, yr_train).coef_)
    lasso_coefs.append(Lasso(alpha=a).fit(Xr_train_scaled, yr_train).coef_)

fig, axes = plt.subplots(1, 2, figsize=(12,4))
axes[0].plot(alphas, ridge_coefs); axes[0].set_xscale('log'); axes[0].set_title('Ridge: coefficients shrink smoothly')
axes[1].plot(alphas, lasso_coefs); axes[1].set_xscale('log'); axes[1].set_title('Lasso: coefficients hit exactly zero')
for ax in axes:
    ax.set_xlabel('alpha (log scale)'); ax.set_ylabel('coefficient value')
plt.tight_layout()
plt.show()

## 7. Advanced Encoding: Target/Mean Encoding
**Why:** OneHotEncoding explodes into too many columns for HIGH-CARDINALITY categoricals (e.g. zip
codes, product IDs with 1000s of values). Target encoding replaces each category with the mean of the
target variable for that category — compact and often more predictive, but risks leakage if not done
carefully (must use cross-fold encoding, not encode using the full training set's own target).

In [ ]:
if CE_AVAILABLE:
    # Target encoding demo on 'embarked' (low cardinality here, but same technique scales to high cardinality)
    target_enc = ce.TargetEncoder(cols=['embarked'])
    titanic_te = titanic.copy()
    titanic_te['embarked_target_enc'] = target_enc.fit_transform(titanic_te['embarked'], titanic_te['survived'])
    print(titanic_te[['embarked','embarked_target_enc','survived']].drop_duplicates('embarked'))
    print("\n⚠️ In practice: fit target encoder ONLY on training folds (e.g. inside cross_val or a Pipeline)")
    print("to avoid leaking target information into the encoding.")
else:
    print("Skipping - install category_encoders to run this cell")

## 8. Proper Pipelines with ColumnTransformer
**Why:** real datasets mix numeric and categorical columns that need DIFFERENT preprocessing.
`ColumnTransformer` lets you apply different transformers to different columns, all inside one
`Pipeline` — this is the "correct", leakage-safe, production-ready way to preprocess data (each
transformer is fit only on training folds during cross-validation).

In [ ]:
numeric_features = ['age', 'fare', 'sibsp', 'parch']
categorical_features = ['sex', 'embarked', 'pclass']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
])

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=200, random_state=42))
])

Xt_train, Xt_test, yt_train, yt_test = train_test_split(
    titanic[numeric_features + categorical_features], titanic['survived'], test_size=0.2, random_state=42, stratify=titanic['survived'])

full_pipeline.fit(Xt_train, yt_train)
print("ColumnTransformer Pipeline accuracy:", accuracy_score(yt_test, full_pipeline.predict(Xt_test)))
print("\nThis single object handles raw mixed data in -> predictions out. Ideal for deployment.")

## 9. Model Interpretability with SHAP
**Why:** "Why did the model predict THIS for THIS specific passenger?" Feature importances (from
Section 12 of notebook 2) only tell you GLOBAL importance. **SHAP** (SHapley Additive exPlanations)
gives per-PREDICTION explanations — how much each feature pushed a specific prediction up or down —
based on game theory. Essential for explaining black-box models to stakeholders or debugging.

In [ ]:
if SHAP_AVAILABLE:
    rf_shap = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)
    explainer = shap.TreeExplainer(rf_shap)
    shap_values = explainer.shap_values(X_test)

    # Summary plot - global view: which features matter most and how (color = feature value)
    shap.summary_plot(shap_values[:,:,1] if len(np.array(shap_values).shape)==3 else shap_values,
                       X_test, show=True)
else:
    print("Skipping - install shap to run this cell")

## 10. Extra Clustering Algorithms

### 10.1 Spectral Clustering
**Idea:** builds a similarity graph between points, then clusters based on the graph's structure
(eigenvectors of the graph Laplacian). Handles non-convex shapes well, like DBSCAN, but requires
specifying K (unlike DBSCAN) and doesn't scale well to very large datasets.

In [ ]:
from sklearn.datasets import make_moons
X_moons, _ = make_moons(n_samples=300, noise=0.07, random_state=42)
X_moons_s = StandardScaler().fit_transform(X_moons)

spectral = SpectralClustering(n_clusters=2, affinity='nearest_neighbors', random_state=42)
labels_spectral = spectral.fit_predict(X_moons_s)

plt.figure(figsize=(5,4))
plt.scatter(X_moons_s[:,0], X_moons_s[:,1], c=labels_spectral, cmap='viridis')
plt.title('Spectral Clustering on moons (handles non-convex shapes)')
plt.show()

### 10.2 Mean Shift Clustering
**Idea:** iteratively shifts points toward the densest nearby region (mode-seeking). Like DBSCAN,
it doesn't require specifying K upfront — the number of clusters is discovered automatically based
on a `bandwidth` parameter (neighborhood size).

In [ ]:
from sklearn.datasets import make_blobs
X_blobs, _ = make_blobs(n_samples=300, centers=4, cluster_std=0.8, random_state=42)
X_blobs_s = StandardScaler().fit_transform(X_blobs)

bandwidth = estimate_bandwidth(X_blobs_s, quantile=0.2)
mean_shift = MeanShift(bandwidth=bandwidth)
labels_ms = mean_shift.fit_predict(X_blobs_s)

plt.figure(figsize=(5,4))
plt.scatter(X_blobs_s[:,0], X_blobs_s[:,1], c=labels_ms, cmap='viridis')
plt.scatter(mean_shift.cluster_centers_[:,0], mean_shift.cluster_centers_[:,1], c='red', marker='X', s=200)
plt.title(f'Mean Shift (auto-found {len(set(labels_ms))} clusters)')
plt.show()

## 11. Factor Analysis & ICA (alternatives to PCA)

**Factor Analysis:** like PCA, but assumes observed features are noisy linear combinations of a
smaller number of unobserved "latent factors" — models noise explicitly, common in psychometrics/
social science for finding underlying constructs (e.g. "general intelligence" from test scores).

**ICA (Independent Component Analysis):** finds components that are STATISTICALLY INDEPENDENT
(not just uncorrelated like PCA) — classic use case is separating mixed audio signals
("cocktail party problem") or separating mixed independent source signals in general.

In [ ]:
iris = sns.load_dataset('iris')
X_iris_s = StandardScaler().fit_transform(iris.drop(columns=['species']))

fa = FactorAnalysis(n_components=2, random_state=42)
X_fa = fa.fit_transform(X_iris_s)

ica = FastICA(n_components=2, random_state=42)
X_ica = ica.fit_transform(X_iris_s)

fig, axes = plt.subplots(1, 2, figsize=(11,4))
sns.scatterplot(x=X_fa[:,0], y=X_fa[:,1], hue=iris['species'], ax=axes[0], legend=False)
axes[0].set_title('Factor Analysis (2 latent factors)')
sns.scatterplot(x=X_ica[:,0], y=X_ica[:,1], hue=iris['species'], ax=axes[1], legend=False)
axes[1].set_title('ICA (2 independent components)')
plt.tight_layout()
plt.show()

## 12. Data Leakage — A Hands-On Demonstration
**Why this matters:** data leakage is when information from outside the training set (often,
indirectly, the target itself) sneaks into your features, giving unrealistically good validation
scores that COLLAPSE in production. Below is a deliberate example of the mistake, then the fix.

In [ ]:
# ❌ WRONG: scaling BEFORE train-test split leaks test set statistics into training
X_full = titanic_enc.drop(columns=['survived'])
y_full = titanic_enc['survived']

scaler_leaky = StandardScaler()
X_full_scaled_leaky = scaler_leaky.fit_transform(X_full)   # fit on EVERYTHING including future test set
Xl_train, Xl_test, yl_train, yl_test = train_test_split(X_full_scaled_leaky, y_full, test_size=0.2, random_state=42)

leaky_model = LogisticRegression(max_iter=1000).fit(Xl_train, yl_train)
print("Leaky pipeline accuracy:", accuracy_score(yl_test, leaky_model.predict(Xl_test)))

# ✅ CORRECT: split FIRST, fit scaler only on training data
Xc_train, Xc_test, yc_train, yc_test = train_test_split(X_full, y_full, test_size=0.2, random_state=42)
scaler_correct = StandardScaler().fit(Xc_train)          # fit ONLY on training data
Xc_train_scaled = scaler_correct.transform(Xc_train)
Xc_test_scaled = scaler_correct.transform(Xc_test)

correct_model = LogisticRegression(max_iter=1000).fit(Xc_train_scaled, yc_train)
print("Correct pipeline accuracy:", accuracy_score(yc_test, correct_model.predict(Xc_test_scaled)))
print("\nNote: the gap may be small on this dataset, but on smaller/noisier data leakage can")
print("inflate validation scores dramatically while production performance quietly craters.")

## 13. Minimal Deployment Example (Flask API)
**Why:** a trained model is only useful once it can serve predictions. Below is a minimal Flask app
skeleton showing how to load a saved model and expose it as an API endpoint. This is NOT meant to be
run inside the notebook — copy it into a separate `app.py` file and run with `python app.py`.

In [ ]:

# --- Save this as app.py and run separately (not inside the notebook) ---
"""
from flask import Flask, request, jsonify
import joblib
import pandas as pd

app = Flask(__name__)
model = joblib.load('best_classification_model.pkl')  # from notebook 2, Section 16

@app.route('/predict', methods=['POST'])
def predict():
    # Expects JSON body matching the training feature columns
    data = request.get_json()
    df = pd.DataFrame([data])
    prediction = model.predict(df)[0]
    probability = model.predict_proba(df)[0].tolist() if hasattr(model, 'predict_proba') else None
    return jsonify({'prediction': int(prediction), 'probability': probability})

if __name__ == '__main__':
    app.run(debug=True, port=5000)

# Test with:
# curl -X POST http://127.0.0.1:5000/predict -H "Content-Type: application/json" \
#      -d '{"age": 29, "fare": 32.5, "sibsp": 0, "parch": 0, "pclass": 1, "sex_male": 0, "embarked_Q": 0, "embarked_S": 1}'
"""
print("See the commented block above - copy into a standalone app.py to actually run the API.")


## 14. Summary — Topics Covered in This Notebook
- Neural network basics: MLPClassifier, MLPRegressor
- Naive Bayes variants: GaussianNB, MultinomialNB, BernoulliNB (and when to use each)
- Threshold tuning (optimizing decision cutoff beyond default 0.5)
- Probability calibration: calibration curves, Brier score, CalibratedClassifierCV
- Statistical significance testing between models (paired t-test on CV folds)
- Regularization path visualization (Ridge vs Lasso coefficient shrinkage)
- Advanced encoding: Target/Mean Encoding (and the leakage risk)
- ColumnTransformer + Pipeline for mixed numeric/categorical data (production-ready pattern)
- Model interpretability: SHAP values (global summary + per-prediction explanations)
- Extra clustering: Spectral Clustering, Mean Shift
- Factor Analysis and ICA (alternatives to PCA)
- Hands-on data leakage demonstration (wrong vs correct scaling order)
- Minimal deployment example (Flask API skeleton)

## 🎓 Complete Set — All 4 Notebooks
1. `01_Supervised_Regression.ipynb` — core regression algorithms & workflow
2. `02_Supervised_Classification.ipynb` — core classification algorithms & workflow
3. `03_Unsupervised_Learning.ipynb` — clustering, dimensionality reduction, anomaly detection, association rules
4. `04_Advanced_Topics.ipynb` — neural nets, calibration, interpretability, leakage, deployment, and everything else

Together these span the practical ML curriculum for tabular data end-to-end.
